In [ ]:
# Option 1: Using torchvision's pretrained models as feature extractors (encoder part)
def load_pretrained_encoder():
    # Load a pretrained ResNet but remove the final classification layer
    model = resnet18(pretrained=True)
    # Remove the final fully connected layer
    encoder = nn.Sequential(*list(model.children())[:-1])
    # This gives you a 512-dimensional feature vector for each image
    return encoder

# Option 2: Using pytorch-pretrained-vae package (if installed)
def load_pretrained_vae():
    try:
        import pytorch_pretrained_vae as vae_models
        model = vae_models.VAE_MNIST()  # For MNIST
        # Or other models like:
        # model = vae_models.VAE_CIFAR10()
        # model = vae_models.BetaVAE()
        return model
    except ImportError:
        print("pytorch_pretrained_vae package not installed.")
        return None

# Option 3: Using a torchvision-like API for pretrained models from HuggingFace
def load_stable_diffusion_vae():
    try:
        from diffusers import AutoencoderKL
        
        # Load the stable diffusion VAE
        vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
        return vae
    except ImportError:
        print("diffusers package not installed. Run 'pip install diffusers transformers'")
        return None

# For MNIST specifically - a much simpler approach with HuggingFace
def load_mnist_vae():
    try:
        from diffusers import AutoencoderKL
        # This is just an example path - you'd need the actual pretrained model path
        model_path = "fusing/vae-mnist"
        vae = AutoencoderKL.from_pretrained(model_path)
        return vae
    except (ImportError, OSError):
        print("Model not available or diffusers package not installed.")
        return None

# Example usage:
if __name__ == "__main__":
    # 1. If you want to use a pretrained CNN as an encoder
    encoder = load_pretrained_encoder()
    
    # 2. If you want a full VAE for MNIST
    vae = load_stable_diffusion_vae()
    
    # Example of encoding an image with Stable Diffusion VAE
    if vae is not None:
        # Create a random image batch (3, 512, 512)
        sample_image = torch.randn(1, 3, 512, 512)
        
        # Encode the image to latent space
        with torch.no_grad():
            latent = vae.encode(sample_image).latent_dist.sample()
            print(f"Latent shape: {latent.shape}")  # Should be [1, 4, 64, 64]
            
            # Decode back to image space
            decoded = vae.decode(latent).sample
            print(f"Decoded shape: {decoded.shape}")  # Should be [1, 3, 512, 512]

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load MNIST from OpenML
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist.data, mnist.target
X /= 255.0  

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load MNIST dataset from OpenML
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist.data, mnist.target.astype(int)  # Convert labels to integers

# Normalize pixel values to [0,1]
X /= 255.0  

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a logistic regression model
clf = LogisticRegression(max_iter=1000, solver='lbfgs')
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Find misclassified examples
misclassified_indices = np.where(y_pred != y_test)[0]

# Visualize some misclassified images
num_images = 10  # Number of misclassified images to show
plt.figure(figsize=(10, 5))

for i, idx in enumerate(misclassified_indices[:num_images]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test.iloc[idx].values.reshape(28, 28), cmap="gray")
    plt.title(f"Pred: {y_pred[idx]}, True: {y_test.iloc[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=len(trainset), shuffle=True)  # Full batch like sklearn
testloader = torch.utils.data.DataLoader(testset, batch_size=len(testset), shuffle=False)

# Define the logistic regression model (no hidden layers)
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(28 * 28, 10)  # MNIST has 10 classes

    def forward(self, x):
        x = x.view(-1, 28 * 28)  # Flatten image
        return self.linear(x)  # No activation (CrossEntropyLoss applies softmax)

# Initialize model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LogisticRegressionModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Similar to lbfgs behavior

# Training loop
num_epochs = 1000  # Matches sklearn's max_iter
for epoch in range(num_epochs):
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluate on test set
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

# Compute accuracy
accuracy = (predicted == labels).float().mean().item()
print(f"Accuracy: {accuracy:.4f}")

# Identify misclassified examples
misclassified_indices = (predicted != labels).nonzero(as_tuple=True)[0]

# Visualize misclassified images
num_images = 10  # Show first 10 misclassified images
plt.figure(figsize=(10, 5))

for i, idx in enumerate(misclassified_indices[:num_images].cpu().numpy()):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[idx].cpu().view(28, 28), cmap="gray")
    plt.title(f"Pred: {predicted[idx].item()}, True: {labels[idx].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True)  # Use mini-batches
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False)

# Define the logistic regression model
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(28 * 28, 10)  # MNIST has 10 classes

    def forward(self, x):
        x = x.view(-1, 28 * 28)  # Flatten image
        return self.linear(x)  # No activation (CrossEntropyLoss applies softmax)

# Initialize model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LogisticRegressionModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam is better for small datasets

# Training loop (Reduced epochs + Mini-batches)
num_epochs = 10  # Faster training
for epoch in range(num_epochs):
    running_loss = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}')

# Evaluate on test set
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.append(predicted)
        all_labels.append(labels)

# Convert predictions to a single tensor
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

# Compute accuracy
accuracy = (all_preds == all_labels).float().mean().item()
print(f"Accuracy: {accuracy:.4f}")

# Identify misclassified examples
misclassified_indices = (all_preds != all_labels).nonzero(as_tuple=True)[0]

# Visualize misclassified images
num_images = 10  # Show first 10 misclassified images
plt.figure(figsize=(10, 5))

for i, idx in enumerate(misclassified_indices[:num_images].cpu().numpy()):
    plt.subplot(2, 5, i + 1)
    plt.imshow(testset.data[idx].cpu().numpy(), cmap="gray")
    plt.title(f"Pred: {all_preds[idx].item()}, True: {all_labels[idx].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()
